In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Cell 1: load dataset
df = pd.read_parquet("../data/processed/ca1_long.parquet")
print("shape:", df.shape)
print("price missing:", df["sell_price"].isna().sum())

shape: (5918109, 19)
price missing: 1129842


In [15]:
# Cell 2: Horizon, final test window and backtest folds
H = 28
LAST_DAY = int(df["d_num"].max())
TEST_START = LAST_DAY - H + 1

rows = []
for i in (1, 2, 3):
    val_end = TEST_START - 1 -(3 - i) * H
    val_start = val_end - H + 1
    rows.append({"fold": i, "train_end": val_start - 1,
                 "val_start": val_start, "val_end": val_end})
folds = pd.DataFrame(rows)

date_of = df.drop_duplicates("d_num").set_index("d_num")["date"]
folds["val_start_date"] = folds["val_start"].map(date_of).dt.date
folds["val_end_date"] = folds["val_end"].map(date_of).dt.date

display(folds)

print("final test window:", TEST_START, "to", LAST_DAY,
      "|", date_of[TEST_START].date(), "to", date_of[LAST_DAY].date())



,fold,train_end,val_start,val_end,val_start_date,val_end_date
0,1,1829,1830,1857,2016-02-01,2016-02-28
1,2,1857,1858,1885,2016-02-29,2016-03-27
2,3,1885,1886,1913,2016-03-28,2016-04-24


final test window: 1914 to 1941 | 2016-04-25 to 2016-05-22


In [5]:
# Cell 3: Drop pre-launch rows

first_day = (df.loc[df["sell_price"].notna()]
             .groupby("item_id", observed=True)["d_num"].min().rename("first_day"))

df = df.merge(first_day, on="item_id", how="left")
df = df[df["d_num"] >= df["first_day"]].reset_index(drop=True)

print("rows kept      :", len(df))
print("missing prices :", df["sell_price"].isna().sum())
print("items          :", df["item_id"].nunique())

rows kept      : 4788267
missing prices : 0
items          : 3049


In [8]:
# Cell 4: Item x day matrix
df["sales"] = df["sales"].astype("float32")

Y = df.pivot(index="item_id", columns="d_num", values="sales")
Y = Y.reindex(columns=range(1, LAST_DAY + 1))

print("Y shape          :", Y.shape)
print("NaN cells        :", int(Y.isna().sum().sum()))
print("total units      :", int(Y.sum().sum()), "| long table:", int(df["sales"].sum()))
print("first day metrics:", (Y.notna().idxmax(axis=1).values ==
                              first_day.reindex(Y.index).values).all())
print("memory (MB)      :", round(Y.memory_usage(deep=True).sum() / 1e6, 1))

Y shape          : (3049, 1941)
NaN cells        : 1129842
total units      : 7832248 | long table: 7832248
first day metrics: True
memory (MB)      : 24.0


In [9]:
# Cell 5: Metric functions
def _valid(y, p):
    m = ~np.isnan(y) & ~np.isnan(p)
    return y[m], p[m]

def mae(y, p):
    y, p = _valid(y, p)
    return float(np.abs(y -p).mean())

def rmse(y, p):
    y, p = _valid(y, p)
    return float(np.sqrt(((y - p) ** 2).mean()))

def wape(y, p):
    y, p = _valid(y, p)
    return float(np.abs(y - p).sum() / y.sum())

def mase_scale(Y_train, m = 7):
    """Per-item mean absolute error of the seasonanl naive forecast on the training data."""
    d = np.abs(Y_train[:, m:] - Y_train[:, :-m])
    with np.errstate(all = "ignore"):
        return np.nanmean(d, axis = 1)

def mase(y, p, scale):
    """Mean over items of (item MAE / item scale). y, p: items x days."""
    with np.errstate(all="ignore"):
        item_mae = np.nanmean(np.abs(y - p), axis = 1)
        ratio = item_mae / scale
    ok = np.isfinite(ratio) & (scale > 0)
    return float(ratio[ok].mean()), int ((~ok).sum())


# Sanity checks on tiny made-up data

y = np.array([[0., 2., 4.], [1., 1., 1.]])
print("perfect forecast -> MAE, WAPE:", mae(y, y), wape(y, y))
print("forecast of zeros -> WAPE    :", wape(y, np.zeros_like(y)))
y_nan = np.array([[np.nan, 2., 4.]])
print("NaN actual ignored -> MAE    :", mae(y_nan, np.array([[9., 2., 4.]])))

perfect forecast -> MAE, WAPE: 0.0 0.0
forecast of zeros -> WAPE    : 1.0
NaN actual ignored -> MAE    : 0.0


In [12]:
# Cell 6: The three baselines
import warnings
def _finish(fc):
    return np.nan_to_num(fc, nan=0.0)     # items with no history -> forecast 0

def naive(Y_train, h = H):
    last = Y_train[:, -1]
    return _finish(np.repeat(last[:, None], h, axis=1))

def seasonal_naive(Y_train, h=H, m=7):
    last_week = Y_train[:, -m:]
    reps = int(np.ceil(h / m))
    return _finish(np.tile(last_week, (1, reps))[:, :h])

def moving_average(Y_train, h = H, w = 28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        ma = np.nanmean(Y_train[:, -w:], axis=1)
    return _finish(np.repeat(ma[:, None], h, axis=1))

# Sanity check on a toy item with two full weeks
toy = np.array([[1, 2, 3, 4, 5, 6, 7, 1, 2, 3, 4, 5, 6, 7]], dtype="float32")
print("naive          :", naive(toy, h=10)[0])
print("seasonal naive :", seasonal_naive(toy, h=10)[0])
print("moving avg (7) :", moving_average(toy, h=10, w=7)[0])


# shape check on real data (fold 1)
Y_np = Y.to_numpy()
f = folds.iloc[0]
Y_train = Y_np[:, : int(f["train_end"])]
Y_val = Y_np[:, int(f["val_start"]) -1: int(f["val_end"])]
print("train:", Y_train.shape, "| val:", Y_val.shape)
for fn in (naive, seasonal_naive, moving_average):
    print(fn.__name__, "->", fn(Y_train).shape)

naive          : [7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]
seasonal naive : [1. 2. 3. 4. 5. 6. 7. 1. 2. 3.]
moving avg (7) : [4. 4. 4. 4. 4. 4. 4. 4. 4. 4.]
train: (3049, 1829) | val: (3049, 28)
naive -> (3049, 28)
seasonal_naive -> (3049, 28)
moving_average -> (3049, 28)


### **What this three baselines do:**

- Each baseline takes the training matrix (all days up to train_end) and returns a forecast for the next 28 days (items x 28). It sees nothing from the validation window.
- Naive: every future day equals the last observed day. With a 28-day horizon this is a weak yardstick, and on a zero-heavy series it is noisy.
- Seasonal naive (7 days): the last 7 days repeat four times, so a Monday forecast copies the last Monday. The EDA showed the weekday cycle is the strongest pattern, so this should be hard to beat. It is also the scale used in MASE.
- Moving average (28 days): the mean of the last 28 days, flat for all 28 days. It ignores the weekday cycle.
- Items with no history at train_end (not launched yet) would get NaN forecasts, and the metrics would silently skip them. So we fill remaining NaN with 0, a simple rule applied to every baseline. A later model with no history would have to predict something too.

In [17]:
# Cell 7: The backtest function
def backtest(forcaster, name, Y_all = Y_np, folds = folds, h=H, **kw):
    rows = []
    for _, f in folds.iterrows():
        te, vs, ve = int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        assert vs == te + 1 and ve - vs + 1 == h, "fold windows are not consecutive"

        Y_train = Y_all[:, :te]     # the forecaster sees only the past
        Y_val = Y_all[:, vs -1 : ve] # actuals, used only for scoring
        assert Y_train.shape[1] == te

        P = forcaster(Y_train, h=h, **kw)
        assert P.shape == Y_val.shape, f"forecast shape {P.shape} != {Y_val.shape}"

        scale = mase_scale(Y_train)
        m, skipped = mase(Y_val, P, scale)
        rows.append({"model": name, "fold": int(f["fold"]),
                     "MAE": mae(Y_val, P), "RMSE": rmse(Y_val, P),
                     "WAPE": wape(Y_val, P), "MASE": m,
                     "mase_items_skipped": skipped})
    return pd.DataFrame(rows)

def zeros(Y_train, h=H):
    return np.zeros((Y_train.shape[0], h), dtype="float32")

test = backtest(zeros, "zeros")
display(test.round(3))
print("WAPE is 1.0 in every fold:", np.allclose(test["WAPE"], 1.0))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23668\2567910608.py:22: RuntimeWarning: Mean of empty slice
  return np.nanmean(d, axis = 1)


,model,fold,MAE,RMSE,WAPE,MASE,mase_items_skipped
0,zeros,1,1.439,3.768,1.0,0.892,1
1,zeros,2,1.467,3.731,1.0,0.923,0
2,zeros,3,1.499,3.783,1.0,0.945,0


WAPE is 1.0 in every fold: True


### **What this above function do:**
- one function runs any forecaster over the 3 folds and reports MAE, RMSE, WAPE and MASE per fold. Baselines now, LightGBM later, all through the same function, so every comparison is fair.
- A forecaster is a function that takes Y_train (items x days) and returns a forecast of shape (items x 28).
- Leakage guard: the forecaster only receives Y_train, cut at train_end. It never sees the validation window, and an assert checks that the window starts the day after training ends.
- The MASE scale is computed from the training data only.
- We test the function with a forecaster that always predicts 0. Its WAPE must be exactly 1.0 in every fold, which proves the metrics and the slicing agree.

In [18]:
# Cell 8: Run the baselines through the backtest
res = pd.concat([
    backtest(zeros,     "zeros (reference)"),
    backtest(naive,     "naive"),
    backtest(seasonal_naive, "seasonal naive (7d)"),
    backtest(moving_average, "moving average (28d)"),
], ignore_index=True)

summary = (res.groupby("model", sort=False)[["MAE", "RMSE", "WAPE", "MASE"]].mean().round(3))
display(summary)

print("per fold:")
display(res.pivot(index="model", columns="fold", values="WAPE").round(3))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23668\2567910608.py:22: RuntimeWarning: Mean of empty slice
  return np.nanmean(d, axis = 1)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_23668\2567910608.py:22: RuntimeWarning: Mean of empty slice
  return np.nanmean(d, axis = 1)


,MAE,RMSE,WAPE,MASE
model,,,,
zeros (reference),1.468,3.760,1.000,0.920
naive,1.533,3.393,1.045,1.161
seasonal naive (7d),1.288,2.791,0.877,1.022
moving average (28d),1.099,2.257,0.748,0.895


per fold:


fold,1,2,3
model,,,
moving average (28d),0.747,0.753,0.745
naive,1.091,1.112,0.932
seasonal naive (7d),0.873,0.869,0.888
zeros (reference),1.000,1.000,1.000


In [20]:
# Cell 9: Error at item level vs store total (hierarchy)
cat_arr = (df.drop_duplicates("item_id").set_index("item_id")["cat_id"]
           .reindex(Y.index).astype(str).to_numpy())
cats = sorted(set(cat_arr))

models = {"zeros (reference)": zeros, "naive": naive,
          "seasonal naive (7d)": seasonal_naive, "moving average (28d)": moving_average}
def wape_arr(a, f):
    return float(np.abs(a - f).sum() / a.sum())

rows = []
for name, fn in models.items():
    for _, f in folds.iterrows():
        te, vs, ve = int(f["train_end"]), int(f["val_start"]), int(f["val_end"])
        Yt, Yv = Y_np[:, :te], Y_np[:, vs - 1 : ve]
        P = fn(Yt, h=H)
        P = np.where(np.isnan(Yv), np.nan, P)    # same cells as the actuals

        row = {"model": name, "fold": int(f["fold"])}
        row["item-day"] = wape(Yv, P)

        a_c = np.vstack([np.nansum(Yv[cat_arr == c], axis=0) for c in cats])
        p_c = np.vstack([np.nansum(P[cat_arr == c], axis=0) for c in cats])
        row["category-day"] = wape_arr(a_c, p_c)

        row["store-day"] = wape_arr(np.nansum(Yv, axis=0), np.nansum(P, axis=0))

        for c in cats:
            row[f"item-day {c}"] = wape(Yv[cat_arr == c], P[cat_arr == c])
        rows.append(row)

hier = pd.DataFrame(rows)
display(hier.groupby("model", sort=False).mean(numeric_only=True).drop(columns="fold").round(3))

,item-day,category-day,store-day,item-day FOODS,item-day HOBBIES,item-day HOUSEHOLD
model,,,,,,
zeros (reference),1.000,1.000,1.000,1.000,1.000,1.000
naive,1.045,0.338,0.323,0.979,1.299,1.109
seasonal naive (7d),0.877,0.119,0.102,0.805,1.162,0.941
moving average (28d),0.748,0.171,0.164,0.689,0.971,0.806


## Summary of Notebook 03: baselines and backtest

**Setup**
- Horizon 28 days; final test window d_1914-d_1941 (2016-04-25 to 2016-05-22), untouched until Notebook 10
- 3 backtest folds, each 4 weeks (Mon-Sun), 2016-02-01 to 2016-04-24
- Pre-launch rows dropped (no price = item not yet on sale); metrics skip those cells
- Metrics: MAE, RMSE, WAPE (headline), MASE (7-day seasonal scale)

**Results (mean of 3 folds, item-day level)**

| Model | MAE | RMSE | WAPE | MASE |
|---|---|---|---|---|
| zeros (reference) | 1.468 | 3.760 | 1.000 | 0.920 |
| naive | 1.533 | 3.393 | 1.045 | 1.161 |
| seasonal naive (7d) | 1.288 | 2.791 | 0.877 | 1.022 |
| **moving average (28d)** | **1.099** | **2.257** | **0.748** | **0.895** |

**Hierarchy (WAPE):** moving average 0.748 (item-day) -> 0.171 (category-day) -> 0.164 (store-day); seasonal naive 0.877 -> 0.119 -> 0.102

**Findings**
- Bar to beat: WAPE 0.748 at item-day level (moving average, stable at 0.745-0.753 in all folds)
- Seasonal naive wins at category and store level but loses to the moving average per item: the weekday cycle is real, item-level data is too noisy to copy one week
- Seasonal naive is worse than predicting zero for HOBBIES (1.162)
- WAPE and MASE rank zeros vs seasonal naive differently: MASE averages per-item ratios, so slow items count as much as busy ones; WAPE is the headline

**Limits (for the README)**
- All folds are February-April; no summer or holiday fold
- Zero-sales days may include stockouts; "demand" means observed sales
- MASE is a mean of per-item ratios and is unstable for very slow items